
# Chapter 01: Circular Data

Source span: printed pages 1-12; PDF pages 20-31. The source was inspected for topic order and terminology only. The examples, diagrams, computations, and checks in this notebook are original and synthetic.

## Chapter Goal

Circular data begin with a deceptively simple decision: are the observations points on a circle, axes with opposite directions identified, clock times obtained by wrapping a line, or compass directions obtained by projecting planar vectors? Those choices determine which displays and summaries are honest. A histogram that works on the line can mislead after a cut point is chosen. A rose diagram must encode frequency by sector area rather than just radius. Axial observations must identify `theta` and `theta + pi`, so doubling angles is a modeling move, not a plotting trick.

This notebook builds the chapter from the geometry outward. We construct clock and compass coordinate conventions, compare circular raw plots with circular histograms, rose diagrams, and unrolled histograms, show how axial doubling removes the sign ambiguity, and contrast wrapped time data with projected compass data. The final checks record the invariants that make a display faithful: counts are preserved, rose sector area is proportional to frequency, doubled antipodal axes agree, and circular distance ignores the arbitrary cut point.



## Visualization Storyboard And Library Routing

- **Coordinate convention panel.** Clock data and compass data use different zero directions and orientations. A Matplotlib diagram is durable and lets labels carry the convention explicitly.
- **Display comparison dashboard.** The same synthetic circular sample is shown as raw points, a circular histogram, a rose diagram, and an unrolled histogram with two different cut points. The inspection target is how the cut point can create or remove an artificial split.
- **Axial doubling diagram.** A small set of axes is displayed before and after doubling. The check confirms that opposite directions become identical after doubling.
- **Wrapping versus projecting lab.** Plotly shows two generative stories: line values wrapped modulo a period, and planar vectors projected radially to the unit circle. Interaction is useful here because the reader can inspect the same angle as a line residue or as a vector direction.

The notebook uses NumPy for angle arithmetic, Matplotlib for static circular diagrams, Plotly for the interactive generative comparison, and course-local artifact helpers for saved outputs and checks.


In [ ]:

from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_dirstats_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate Directional-Statistics course root")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "chapter-01"
source_span = {"printed": "1-12", "pdf": "20-31"}
print(f"Course root: {BOOK_ROOT}")
source_span


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.validation import assert_artifacts

rng = np.random.default_rng(101)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 130, "axes.grid": True, "grid.alpha": 0.25})


def wrap_2pi(theta):
    return np.mod(theta, 2.0 * np.pi)


def wrap_pi(theta):
    return (theta + np.pi) % (2.0 * np.pi) - np.pi


def circular_distance(a, b):
    return np.abs(wrap_pi(a - b))


def polar_hist_counts(theta, bins=12, offset=0.0):
    edges = np.linspace(0.0, 2.0 * np.pi, bins + 1) + offset
    shifted = wrap_2pi(theta - offset)
    counts, base_edges = np.histogram(shifted, bins=np.linspace(0, 2*np.pi, bins+1))
    centers = offset + 0.5 * (base_edges[:-1] + base_edges[1:])
    widths = np.diff(base_edges)
    return counts, wrap_2pi(centers), widths


def mean_direction(theta):
    z = np.exp(1j * theta)
    resultant = z.mean()
    return float(np.angle(resultant) % (2*np.pi)), float(abs(resultant))


def make_mixture_angles(n=90):
    primary = rng.vonmises(mu=np.deg2rad(315), kappa=6.0, size=int(0.70*n))
    secondary = rng.vonmises(mu=np.deg2rad(35), kappa=5.0, size=n - len(primary))
    return wrap_2pi(np.concatenate([primary, secondary]))

angles = make_mixture_angles(96)
mean_angle, rbar = mean_direction(angles)



## Clock And Compass Are Different Coordinate Stories

A circular observation is a point on a unit circle after an origin and orientation have been chosen. Clock measurements wrap a line: midnight and 24:00 are the same point. Compass measurements project a planar direction onto the circle: north is often zero and angles are measured clockwise. Both become angles, but they do not have the same generative story.

The first artifact makes the convention explicit. This matters because a sign error or a clockwise/anticlockwise swap can move an entire sample. Later statistics can be invariant to rotating the zero direction, but they are not invariant to confusing a clock with a compass or a direction with an axis.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.0), subplot_kw={"aspect": "equal"})
t = np.linspace(0, 2*np.pi, 400)
for ax, title in zip(axes, ["Clock convention: wrap time", "Compass convention: project direction"]):
    ax.plot(np.cos(t), np.sin(t), color="#345995", linewidth=2)
    ax.scatter([1], [0], color="#2A9D8F", s=70)
    ax.arrow(0, 0, 0.8, 0, head_width=0.05, color="#2A9D8F", length_includes_head=True)
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title)

# Clock: midnight at top in everyday display, but mathematical angle shown from x-axis.
axes[0].text(0, 1.08, "24:00 = 0:00", ha="center", color="#B84A62", fontweight="bold")
axes[0].text(1.04, 0, "math zero", va="center", color="#2A9D8F")
for hour in [3, 6, 9, 12]:
    phi = np.pi/2 - 2*np.pi*(hour % 12)/12
    axes[0].text(1.03*np.cos(phi), 1.03*np.sin(phi), str(hour), ha="center", va="center", fontsize=9)

# Compass: north zero, clockwise positive.
axes[1].arrow(0, 0, 0, 0.82, head_width=0.05, color="#B84A62", length_includes_head=True)
axes[1].text(0, 1.08, "N = 0 deg", ha="center", color="#B84A62", fontweight="bold")
axes[1].text(1.08, 0, "E = 90 deg", ha="center", color="#B84A62")
axes[1].annotate("clockwise positive", xy=(0.75, 0.45), xytext=(0.28, -0.72), arrowprops=dict(arrowstyle="->", color="#555555"), fontsize=9)
fig.suptitle("Circular coordinates need an origin, orientation, and measurement story", y=1.02, fontsize=14)
fig.tight_layout()
convention_path = save_matplotlib(fig, TOPIC, "coordinate-conventions", "clock-compass-coordinate-conventions.png")
plt.close(fig)
display_artifact(convention_path, width=880)

convention_checks = {
    "wrapped_zero_equals_full_turn": bool(np.isclose(wrap_2pi(2*np.pi), 0.0)),
    "periodic_distance_across_cut_degrees": float(np.degrees(circular_distance(np.deg2rad(359), np.deg2rad(1)))),
}
convention_checks



## Four Displays For The Same Circular Sample

A circular raw-data plot shows points without imposing bins. A circular histogram groups angles around the circle. A rose diagram replaces bars by sectors, so the area of each sector should represent frequency. An unrolled linear histogram is useful only after the cut point is chosen with care.

The synthetic sample below has observations near both sides of zero degrees. If the unrolled histogram is cut at zero, the cluster appears split. If the cut is placed opposite the preferred direction, the same data look unimodal. The lesson is not that linear histograms are forbidden. The lesson is that a cut point is an analytical choice.


In [ ]:

counts, centers, widths = polar_hist_counts(angles, bins=12)
rose_radii = np.sqrt(counts / counts.max())

fig = plt.figure(figsize=(13, 9))
ax0 = fig.add_subplot(2, 2, 1, projection="polar")
ax0.scatter(angles, np.ones_like(angles), s=18, alpha=0.75, color="#345995")
ax0.plot([mean_angle, mean_angle], [0, 1.05], color="#B84A62", linewidth=2, label=f"mean, Rbar={rbar:.2f}")
ax0.set_title("Raw circular plot")
ax0.set_yticklabels([])
ax0.legend(loc="lower left", bbox_to_anchor=(-0.1, -0.1), fontsize=8)

ax1 = fig.add_subplot(2, 2, 2, projection="polar")
ax1.bar(centers, counts, width=widths, align="center", color="#8ecae6", edgecolor="#23343b", alpha=0.9)
ax1.set_title("Circular histogram")
ax1.set_yticklabels([])

ax2 = fig.add_subplot(2, 2, 3, projection="polar")
ax2.bar(centers, rose_radii, width=widths, align="center", color="#F4A261", edgecolor="#23343b", alpha=0.9)
ax2.set_title("Rose diagram: radius uses sqrt(count)")
ax2.set_yticklabels([])

ax3 = fig.add_subplot(2, 2, 4)
angles_deg = np.degrees(angles)
angles_cut180 = np.degrees(wrap_2pi(angles - np.pi)) - 180
bins = np.linspace(-180, 180, 13)
ax3.hist(((angles_deg + 180) % 360) - 180, bins=bins, alpha=0.55, label="cut at 0 deg", color="#B84A62")
ax3.hist(angles_cut180, bins=bins, alpha=0.55, label="cut opposite cluster", color="#345995")
ax3.set_title("Unrolled histograms depend on cut point")
ax3.set_xlabel("unrolled angle (degrees)")
ax3.set_ylabel("frequency")
ax3.legend()
fig.suptitle("Display choices for circular data: same counts, different inspection targets", y=1.02, fontsize=14)
fig.tight_layout()
display_path = save_matplotlib(fig, TOPIC, "display-comparison", "circular-display-comparison-dashboard.png")
plt.close(fig)
display_artifact(display_path, width=940)

sector_areas = 0.5 * widths * rose_radii**2
nonzero = counts > 0
area_ratio_spread = float(np.max(sector_areas[nonzero] / counts[nonzero]) - np.min(sector_areas[nonzero] / counts[nonzero]))
display_checks = {
    "histogram_counts_preserve_sample_size": bool(counts.sum() == len(angles)),
    "rose_sector_area_proportional_to_frequency": bool(area_ratio_spread < 1e-12),
    "mean_resultant_between_zero_and_one": bool(0.0 <= rbar <= 1.0),
}
display_checks



## Axial Data: Opposite Directions Are The Same Observation

Axial data record unoriented lines. A crack direction, an optical axis, or an elongation direction may not distinguish `theta` from `theta + 180 degrees`. Doubling angles converts axes into ordinary circular directions because both representatives of the same axis land at the same doubled angle modulo a full turn.

The diagram below shows both views. On the left, each axis is drawn as a line through the circle. On the right, the doubled angles become oriented circular points. The invariant check is the whole point: `2 theta` and `2(theta + pi)` agree modulo `2 pi`.


In [ ]:

axis_angles = np.deg2rad(np.array([12, 28, 52, 96, 118, 145]))
doubled = wrap_2pi(2 * axis_angles)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), subplot_kw={"projection": "polar"})
for theta in axis_angles:
    axes[0].plot([theta, theta + np.pi], [1, 1], color="#345995", linewidth=2)
    axes[0].scatter([theta, theta + np.pi], [1, 1], color="#345995", s=20)
axes[0].set_title("Axial observations: theta equals theta + pi")
axes[0].set_yticklabels([])
axes[1].scatter(doubled, np.ones_like(doubled), color="#E76F51", s=55)
for theta, doubled_theta in zip(axis_angles, doubled):
    axes[1].plot([0, doubled_theta], [0, 1], color="#E76F51", linewidth=1, alpha=0.55)
axes[1].set_title("Doubled angles as circular data")
axes[1].set_yticklabels([])
fig.suptitle("Axial-to-circular conversion by doubling angles", y=1.02, fontsize=14)
fig.tight_layout()
axial_path = save_matplotlib(fig, TOPIC, "axial-data", "axial-doubling-diagram.png")
plt.close(fig)
display_artifact(axial_path, width=880)

axial_checks = {
    "antipodal_axes_agree_after_doubling": bool(np.allclose(wrap_2pi(2*(axis_angles + np.pi)), doubled)),
    "doubled_angles_in_unit_circle_domain": bool(np.all((doubled >= 0) & (doubled < 2*np.pi))),
}
axial_checks



## Wrapping Versus Projecting

The source chapter distinguishes two common ways circular data arise. Clock-like data wrap a linear variable modulo a period. Compass-like data project planar vectors radially onto the unit circle. Both produce angles, but their failure modes differ. Wrapped data have a cut point; projected data can be unstable when the vector length is close to zero.

The interactive lab below places the two stories side by side. The wrapped trace shows a line variable repeatedly folded into `[0, 2 pi)`. The projection panel shows planar vectors and their unit directions. The checks record that wrapping preserves periodic distance and projection produces unit vectors except at the excluded origin.


In [ ]:

time_values = np.linspace(-5, 31, 145)
wrapped_hours = np.mod(time_values, 24.0)
wrapped_angles = 2*np.pi*wrapped_hours/24.0
vectors = rng.normal(size=(45, 2)) + np.array([1.3, 0.45])
lengths = np.linalg.norm(vectors, axis=1)
unit_vectors = vectors / lengths[:, None]
vector_angles = np.arctan2(unit_vectors[:, 1], unit_vectors[:, 0])

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter"}, {"type": "scatter"}]], subplot_titles=("Wrapping a line into clock phase", "Projecting planar vectors to directions"))
fig.add_trace(go.Scatter(x=time_values, y=wrapped_hours, mode="lines", name="hour mod 24", line=dict(color="#345995")), row=1, col=1)
fig.add_trace(go.Scatter(x=time_values, y=24*np.ones_like(time_values), mode="lines", name="period", line=dict(color="#999999", dash="dash")), row=1, col=1)
for vec, unit in zip(vectors, unit_vectors):
    fig.add_trace(go.Scatter(x=[0, vec[0]], y=[0, vec[1]], mode="lines", line=dict(color="rgba(184,74,98,0.28)"), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=[0, unit[0]], y=[0, unit[1]], mode="lines", line=dict(color="rgba(52,89,149,0.55)"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=np.cos(np.linspace(0,2*np.pi,200)), y=np.sin(np.linspace(0,2*np.pi,200)), mode="lines", name="unit circle", line=dict(color="#23343b")), row=1, col=2)
fig.update_xaxes(title_text="linear time", row=1, col=1)
fig.update_yaxes(title_text="wrapped hour", row=1, col=1)
fig.update_xaxes(title_text="x", row=1, col=2, scaleanchor="y2", scaleratio=1)
fig.update_yaxes(title_text="y", row=1, col=2)
fig.update_layout(title="Chapter 01 interactive lab: wrapping and radial projection are different generators", height=560, showlegend=True)
wrap_project_path = save_plotly_html(fig, TOPIC, "interactive", "wrapping-versus-projecting-lab.html", include_plotlyjs=True)
display_artifact(wrap_project_path, width="100%", height=580)

wrap_project_checks = {
    "wrapped_hours_in_domain": bool(np.all((wrapped_hours >= 0) & (wrapped_hours < 24.0))),
    "projection_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(unit_vectors, axis=1) - 1.0))),
    "periodic_distance_23_to_1_hours": float(min(abs(23-1), 24-abs(23-1))),
    "excluded_origin_not_sampled": bool(np.all(lengths > 1e-8)),
}
wrap_project_checks



## Reader Checklist

Before computing a statistic on circular data, identify the observation type.

- If the data are **clock-like**, ask where the period begins and whether a cut point will split a real cluster.
- If the data are **compass-like**, ask which zero direction and orientation were used, and remember that the angle came from a planar direction.
- If the data are **axial**, double angles before applying circular summaries, and interpret the result back as an axis.
- If you draw a **rose diagram**, scale radii so sector area, not radius alone, represents frequency.
- If you unroll to a **linear histogram**, state the cut point and inspect whether the visual conclusion changes when the cut moves.

These habits are the foundation for later chapters. Mean directions, circular moments, tests of uniformity, and circular regression all assume that the sample space was chosen correctly before the first formula appeared.


In [ ]:

numeric_checks = {
    **convention_checks,
    **display_checks,
    **axial_checks,
    **wrap_project_checks,
    "source_span": source_span,
    "sample_size": int(len(angles)),
    "all_boolean_checks_pass": bool(
        all(value for value in convention_checks.values() if isinstance(value, bool))
        and all(value for value in display_checks.values() if isinstance(value, bool))
        and all(value for value in axial_checks.values() if isinstance(value, bool))
        and all(value for value in wrap_project_checks.values() if isinstance(value, bool))
    ),
}
assert numeric_checks["all_boolean_checks_pass"], numeric_checks
assert numeric_checks["periodic_distance_across_cut_degrees"] < 3.0
assert numeric_checks["projection_unit_norm_max_error"] < 1e-12
checks_path = save_json(numeric_checks, TOPIC, "checks", "circular-data-invariants.json")
display_artifact(checks_path)
numeric_checks


## Standalone Reading Notes

The most important habit in this opening chapter is to separate the observation from its coordinate label. The number `10 degrees` is not self-explanatory until the zero direction, orientation, and measurement story are known. A clock time, a compass bearing, and an unoriented axis can all be encoded as angles, but the operations that preserve meaning are different.

When you return to this notebook later, use the display dashboard as a diagnostic tool. If moving the cut point changes the visual conclusion, the conclusion belongs to the display rather than to the data. If an axial sample changes after replacing an angle by its opposite, it was handled as directional data by mistake. These checks are deliberately small because they are meant to be reused before more advanced circular inference.



## Takeaways

Circular data are not ordinary numbers with a trigonometric display. They are observations on a closed sample space. Clock data wrap a line; compass data project planar directions; axial data identify opposite directions. Every display should reveal rather than hide those choices.

The practical lesson is to keep the geometry beside the plot. A raw circular plot shows observations without bins. A circular histogram and rose diagram summarize frequencies around the circle. An unrolled histogram is useful only after choosing and reporting a cut point. Axial doubling is required before applying circular methods, and the interpretation must return to axes afterward. Wrapping and projection can produce the same angle value, but they answer different scientific questions.


In [ ]:

final_sanity = {
    "source_span": source_span,
    "artifacts": assert_artifacts(
        [convention_path, display_path, axial_path, wrap_project_path, checks_path],
        min_bytes=100,
    ),
    "core_checks": {
        "counts_preserved": display_checks["histogram_counts_preserve_sample_size"],
        "rose_area_proportional": display_checks["rose_sector_area_proportional_to_frequency"],
        "axial_antipodes_identified": axial_checks["antipodal_axes_agree_after_doubling"],
        "wrapping_domain_valid": wrap_project_checks["wrapped_hours_in_domain"],
        "projection_unit_vectors": wrap_project_checks["projection_unit_norm_max_error"] < 1e-12,
    },
    "pdf_used_for": "source orientation only; no copied prose, tables, screenshots, page crops, or figures",
    "standalone_contract": "coordinate conventions, circular displays, axial doubling, wrapping/projection lab, and invariant checks",
}
assert all(final_sanity["core_checks"].values()), final_sanity
final_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert final_path.exists() and final_path.stat().st_size > 20
final_sanity
